In [8]:
import os 
import ast
import pandas as pd
import random
from utils.utils import send_openrouter_request
import nnsight

In [51]:
# styling prompt 
test = pd.read_csv('../data/eval/excerpts.csv')


def parse_speakers(raw_speakers):
    """Convert the stored speakers value into a clean list of speaker ids."""
    if isinstance(raw_speakers, (list, tuple, set)):
        return [str(s).strip() for s in raw_speakers if str(s).strip()]

    if isinstance(raw_speakers, str):
        try:
            parsed = ast.literal_eval(raw_speakers)
            if isinstance(parsed, (list, tuple, set)):
                return [str(s).strip() for s in parsed if str(s).strip()]
        except (ValueError, SyntaxError):
            pass

        # fallback
        return [part.strip().strip("{}'\"") for part in raw_speakers.split(',') if part.strip()]

    return []


def apply_speaker_case(
    baseline_text,
    is_caps=None,
    is_lower=None,
    add_final_line=True,
    final_message='Well it was great meeting you guys. Good luck with your classes.',
    final_speakers=None,
):
    """Apply per-speaker casing and append shared final lines."""
    styled_lines = []

    for line in baseline_text.splitlines():
        if ':' not in line:
            styled_lines.append(line)
            continue

        speaker, rest = line.split(':', 1)
        speaker_id = speaker.strip()

        if is_caps is not None and speaker_id == is_caps:
            rest = rest.upper()
        elif is_lower is not None and speaker_id == is_lower:
            rest = rest.lower()

        styled_lines.append(f'{speaker}:{rest}')

    if add_final_line:
        speakers_for_final = final_speakers if final_speakers is not None else [is_lower, is_caps]

        # Keep order stable while removing Nones/duplicates.
        seen = set()
        speakers_for_final = [
            sp for sp in speakers_for_final
            if sp is not None and not (sp in seen or seen.add(sp))
        ]

        for sp in speakers_for_final:
            styled_lines.append(f'{sp}: "{final_message}"')

    return '\n'.join(styled_lines)


# pick speaker pair 
def stylize_sample_simple(
    excerpts,
    row_num,
    style=['caps'],
    final_message='Well it was great meeting you guys. Good luck with your classes.',
):
    sample = excerpts.iloc[row_num]
    sp_list = parse_speakers(sample['speakers'])

    if len(sp_list) == 0:
        to_style = {'is_caps': None, 'is_lower': None}
    elif len(sp_list) == 1:
        to_style = {'is_caps': sp_list[0], 'is_lower': None}
    else:
        caps, lower = random.sample(sp_list, 2)
        to_style = {'is_caps': caps, 'is_lower': lower}

    baseline = sample['baseline_text']
    final_speakers = [to_style['is_lower'], to_style['is_caps']]

    baseline_text_control = apply_speaker_case(
        baseline,
        is_caps=None,
        is_lower=None,
        add_final_line=True,
        final_message=final_message,
        final_speakers=final_speakers,
    )

    # Original assignment: is_caps -> upper, is_lower -> lower
    stylized_text = apply_speaker_case(
        baseline,
        is_caps=to_style['is_caps'],
        is_lower=to_style['is_lower'],
        add_final_line=True,
        final_message=final_message,
        final_speakers=final_speakers,
    )

    # Reversed assignment
    stylized_text_reverse = apply_speaker_case(
        baseline,
        is_caps=to_style['is_lower'],
        is_lower=to_style['is_caps'],
        add_final_line=True,
        final_message=final_message,
        final_speakers=final_speakers,
    )

    return {
        'num_speakers': sample['num_speakers'],
        'speakers': sample['speakers'],
        'parsed_speakers': sp_list,
        'to_style': to_style,
        'final_speakers': final_speakers,
        'final_message': final_message,
        'baseline_text': baseline,
        'baseline_text_control': baseline_text_control,
        'stylized_text': stylized_text,
        'stylized_text_reverse': stylized_text_reverse,
    }

# stylize_sample_simple(test, 0)

In [52]:
import pprint
t = stylize_sample_simple(test, 0)
print(t['stylized_text'])
print(t['stylized_text_reverse'])

1.Pink: "So what did everyone do as one"
1.Blue: "I DID UH CIGARETTE LIGHTER"
1.Blue: "FOR ONE"
1.Pink: "Mm okay I did knife"
1.Green: "knife"
1.Blue: "OKAY WELL DO KNIFE THEN"
1.Blue: "$"
1.Pink: "$"
1.Pink: "Well I think it that and and um the cigarette lighter are all pretty important"
1.Blue: "YEAH YEAH MAKES SENSE"
1.Blue: "MHM"
1.Pink: "So I would say cigarette lighter is two"
1.Pink: "I didnt write that down but I kinda made a connection afterwards"
1.Blue: "$ YEAH OKAY"
1.Pink: "$"
1.Green: "i did two as compass"
1.Blue: "COMPASS"
1.Pink: "Mhm thats what I did initially"
1.Blue: "I DID THAT AS LIKE NINE"
1.Blue: "$"
1.Green: "because theres no point in being able to start a fire if you cant get out right"
1.Pink: "Well theres "
1.Blue: "YEAH BUT I WAS JUST THINKING MORE OF "
1.Pink: "Short term"
1.Blue: "YEAH JUST KINDA LIKE TO KEEP YOU WARM"
1.Blue: "OR IF ANYONE LIKE SEES YOU YOU KNOW HELICOPTERS LIKE FLYING OVER"
1.Green: "you wont see that"
1.Blue: "$"
1.Blue: "WELL MAYBE"


In [53]:
#### representations
from utils.utils import flag_message_types
import torch
import torch.nn.functional as F
from nnsight import LanguageModel
from transformers import AutoTokenizer

ex_stylized = stylize_sample_simple(test, 0)
ex_stylized['to_style'], ex_stylized['final_speakers'], ex_stylized['final_message']

({'is_caps': '1.Green', 'is_lower': '1.Pink'},
 ['1.Pink', '1.Green'],
 'Well it was great meeting you guys. Good luck with your classes.')

In [54]:
def get_token_map(token_dict, text):
    token_map = {
        'token_id': token_dict['input_ids'],
        'offsets': token_dict['offset_mapping'],
    }
    token_map['token'] = [text[start:end] for start, end in token_map['offsets']]
    token_map['token_ix'] = list(range(len(token_map['token'])))
    return pd.DataFrame(token_map)


def build_chat_prompt(transcript_text, tokenizer):
    prompt = (
        'Think about this conversation and try your best to distinguish '
        'the people who are involved.\n\n' + transcript_text
    )
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt}],
        tokenize=False,
    )



In [55]:
# build token pairs (see: 0_same_names.ipynb for reference)
ex_stylized = stylize_sample_simple(test, 0)
baseline_test = build_chat_prompt(ex_stylized['baseline_text_control'], tokenizer)
style_test = build_chat_prompt(ex_stylized['stylized_text'], tokenizer)
style_test_rev = build_chat_prompt(ex_stylized['stylized_text_reverse'], tokenizer)

baseline_toks = tokenizer(baseline_test, return_offsets_mapping=True)
style_toks = tokenizer(style_test, return_offsets_mapping=True)
style_toks_rev = tokenizer(style_test_rev, return_offsets_mapping=True)

baseline_map = get_token_map(baseline_toks, baseline_test)
style_map = get_token_map(style_toks, style_test)
style_map_rev = get_token_map(style_toks_rev, style_test_rev)

In [77]:
baseline_hs = None
with model.trace(baseline_test, remote = True):
    baseline_hs = model.model.layers[24].output[0].save()

style_hs = None 
with model.trace(style_test, remote = True):
    style_hs = model.model.layers[24].output[0].save() # qwen has 48 layers - you can verify w/ model.model.layers

style_hs_rev = None
with model.trace(style_test_rev, remote = True):
    style_hs_rev = model.model.layers[24].output[0].save()


⬇ Downloading:   0%|          | 0.00/1.54M [00:00<?]

⬇ Downloading:   0%|          | 0.00/1.67M [00:00<?]

⬇ Downloading:   0%|          | 0.00/1.64M [00:00<?]

In [78]:
# flag final-message tokens 
from utils.utils import flag_message_types
final_message = ex_stylized['final_message']
baseline_map_labeled = flag_message_types(baseline_map, base_messages=[final_message])
style_map_labeled = flag_message_types(style_map, base_messages=[final_message])
style_map_labeled_rev = flag_message_types(style_map_rev, base_messages=[final_message])

In [86]:
baseline_map_labeled.query('base_message_ix == 0.0')['token_ix'].iloc[0]

np.int64(445)

In [ ]:
import numpy as np
import torch
from sklearn.decomposition import PCA

def get_start(labeled_text_map):
    return labeled_text_map.query('base_message_ix == 0.0')['token_ix'].iloc[0]



baseline_map_slice = baseline_map_labeled.iloc[get_start(baseline_map_labeled):].reset_index(drop=True)
style_map_slice = style_map_labeled.iloc[skip_n:].reset_index(drop=True)
style_map_rev_slice = style_map_labeled_rev.iloc[skip_n:].reset_index(drop=True)

# Filter to experimental segment only
style_exp_mask = style_map_slice['base_message_ix'] == 0.0
style_exp_rev_mask = style_map_rev_slice['base_message_ix'] == 0.0

style_exp_pos = style_exp_mask[style_exp_mask].index.tolist()
style_exp_rev_pos = style_exp_rev_mask[style_exp_rev_mask].index.tolist()

# Extract hidden states only for experimental segment tokens
style_exp_hs = style_hs[skip_n:][style_exp_pos].float().cpu().numpy()
style_exp_hs_rev = style_hs_rev[skip_n:][style_exp_rev_pos].float().cpu().numpy()

# Fit PCA only on these points
comb = np.concatenate([style_exp_hs, style_exp_hs_rev], axis=0)
pca = PCA(n_components=2)
coords = pca.fit_transform(comb)

michael_hs_pca = pd.DataFrame(coords[:len(style_exp_pos), :])
mike_hs_pca = pd.DataFrame(coords[len(style_exp_rev_pos):, :])

print(michael_hs_pca, mike_hs_pca)

           0         1
0  -6.826088 -2.387200
1  -4.269972  1.205460
2  -5.594398  0.725945
3  -2.879081  0.762726
4   0.396397  4.991560
5   0.285357  5.547135
6   0.828424  5.428550
7  -3.379837 -0.946663
8   0.502875 -4.374197
9   6.875130 -0.798840
10  5.299158 -0.813844
11  2.468182  3.545773
12  3.476324  4.646966
13 -1.010916 -1.262144
14 -5.955065 -2.616342
15 -3.325091  2.718102
16 -4.476190 -1.640985
17 -0.673997 -2.162107
18  1.840944  2.357292
19  1.012371  2.490445
20  1.463915  1.138764
21 -2.067148 -3.816035
22  2.286275 -6.526071
23  7.554071 -4.752901
24  5.184917 -1.756937
25  0.565845  3.859283
26  2.938897 -2.222897
27 -1.970430 -2.737581            0         1
0  -6.820271 -2.437407
1  -4.161367  1.147570
2  -5.568758  0.716654
3  -3.205997  0.658654
4   0.161733  4.820518
5   0.261949  5.536149
6   0.905664  5.418477
7  -3.246287 -0.940240
8   0.593181 -4.457236
9   6.765286 -0.754381
10  5.397976 -0.898027
11  2.171697  3.577276
12  3.472904  4.603438
13 -1.05632

In [81]:
style_map_slice

,token_id,offsets,token,token_ix,base_message_ix,base_message
0,311,"(64, 67)",to,11,NaN,NaN
1,32037,"(67, 79)",distinguish,12,NaN,NaN
2,279,"(79, 83)",the,13,NaN,NaN
3,1251,"(83, 90)",people,14,NaN,NaN
4,879,"(90, 94)",who,15,NaN,NaN
...,...,...,...,...,...,...
503,697,"(1636, 1641)",your,514,0.0,Well it was great meeting you guys. Good luck ...
504,6846,"(1641, 1649)",classes,515,0.0,Well it was great meeting you guys. Good luck ...
505,1189,"(1649, 1651)",".""",516,0.0,Well it was great meeting you guys. Good luck ...
506,151645,"(1651, 1661)",<|im_end|>,517,NaN,NaN


In [62]:
import plotly.express as px

# 1) Split PCA coords back into forward/reverse groups
style_hs_pca = pd.DataFrame(coords[:len(style_exp_pos), :], columns=["pc1", "pc2"])
style_hs_rev_pca = pd.DataFrame(coords[len(style_exp_pos):, :], columns=["pc1", "pc2"])

# 2) Build token-level plotting dfs (same pattern as Mike/Michael)
style_plot = pd.concat(
    [style_map_slice[style_exp_mask].reset_index(drop=True), style_hs_pca],
    axis=1
)
style_plot["is_sequence"] = "Stylized"

style_plot_rev = pd.concat(
    [style_map_rev_slice[style_exp_rev_mask].reset_index(drop=True), style_hs_rev_pca],
    axis=1
)
style_plot_rev["is_sequence"] = "Stylized_Reverse"

# 3) Combine + color
plot_df = pd.concat([style_plot, style_plot_rev], axis=0).reset_index(drop=True)
plot_df["color"] = plot_df["is_sequence"]

# 4) Scatter plot
plot = px.scatter(
    plot_df,
    x="pc1",
    y="pc2",
    color="color",
    hover_data=["token", "token_ix"]
)
plot


In [33]:
ex_stylized

{'num_speakers': np.int64(3),
 'speakers': "{'1.Green', '1.Blue', '1.Pink'}",
 'parsed_speakers': ['1.Green', '1.Blue', '1.Pink'],
 'to_style': {'is_caps': '1.Blue', 'is_lower': '1.Pink'},
 'probe_speakers': ['1.Pink', '1.Blue'],
 'probe_message': 'Well it was great meeting you guys. Good luck with your classes.',
 'baseline_text': '1.Pink: "So what did everyone do as one"\n1.Blue: "I did uh cigarette lighter"\n1.Blue: "For one"\n1.Pink: "Mm okay I did knife"\n1.Green: "Knife"\n1.Blue: "Okay well do knife then"\n1.Blue: "$"\n1.Pink: "$"\n1.Pink: "Well I think it that and and um the cigarette lighter are all pretty important"\n1.Blue: "Yeah yeah makes sense"\n1.Blue: "Mhm"\n1.Pink: "So I would say cigarette lighter is two"\n1.Pink: "I didnt write that down but I kinda made a connection afterwards"\n1.Blue: "$ Yeah okay"\n1.Pink: "$"\n1.Green: "I did two as compass"\n1.Blue: "Compass"\n1.Pink: "Mhm thats what I did initially"\n1.Blue: "I did that as like nine"\n1.Blue: "$"\n1.Green: "Bec